# Keystone Eval Analysis — 2026-02-25

Comparing: **claude-haiku**, **claude-opus**, **codex-gpt-5.2**, **codex-mini-gpt-5.1**

---

> **Note:** This notebook is checked in without cell outputs.  
> To execute and save results to a separate file, run:
> ```bash
> uv run jupyter nbconvert --to notebook --execute evals/eda/keystone_eval_analysis.ipynb --output keystone_eval_analysis_executed.ipynb
> ```

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

BASE = Path.home() / "keystone_evals" / "2026-02-26"

records = []
for p in BASE.rglob("eval_result.json"):
    with p.open() as f:
        d = json.load(f)
    rel = p.relative_to(BASE)
    model = rel.parts[0]
    repo = rel.parts[1]
    trial = int(rel.parts[2].replace("trial_", ""))
    br = d.get("bootstrap_result") or {}
    ver = br.get("verification") or {}
    agent = br.get("agent") or {}
    cost = agent.get("cost") or {}
    repo_entry = d.get("repo_entry") or {}
    records.append(
        {
            "model": model,
            "repo": repo,
            "trial": trial,
            "success": d.get("success", False),
            "bootstrap_success": br.get("success", False),
            "verification_success": ver.get("success", False),
            "error_message": br.get("error_message", ""),
            "image_build_seconds": ver.get("image_build_seconds"),
            "test_execution_seconds": ver.get("test_execution_seconds"),
            "agent_duration_seconds": agent.get("duration_seconds"),
            "agent_timed_out": agent.get("timed_out", False),
            "cost_usd": cost.get("cost_usd"),
            "input_tokens": cost.get("token_spending", {}).get("input"),
            "output_tokens": cost.get("token_spending", {}).get("output"),
            "language": repo_entry.get("language", ""),
            "difficulty": repo_entry.get("difficulty", ""),
        }
    )

df = pd.DataFrame(records)
print(
    f"Loaded {len(df)} eval results: {df['model'].nunique()} models x {df['repo'].nunique()} repos"
)
df.head()

Loaded 158 eval results: 4 models x 41 repos


,model,repo,trial,success,bootstrap_success,verification_success,error_message,image_build_seconds,test_execution_seconds,agent_duration_seconds,agent_timed_out,cost_usd,input_tokens,output_tokens,language,difficulty
0,codex-mini-gpt-5.1,flask,0,False,False,False,Build failed: .devcontainer/Dockerfile not found.,NaN,NaN,54.923059,False,0.015853,32855,1064,python,easy
1,codex-mini-gpt-5.1,k-9,0,False,False,False,Build failed: .devcontainer/Dockerfile not found.,NaN,NaN,372.345853,False,0.014042,25307,444,kotlin+java,hard
2,codex-mini-gpt-5.1,bucksamples,0,False,False,False,Build failed: .devcontainer/Dockerfile not found.,NaN,NaN,30.176256,False,0.027903,21554,96,java+c++,medium-hard
3,codex-mini-gpt-5.1,ui,0,False,False,False,Build failed: .devcontainer/Dockerfile not found.,NaN,NaN,52.092701,False,0.031190,57984,839,python+react,medium-hard
4,codex-mini-gpt-5.1,tarantool,0,False,False,False,Build failed: .devcontainer/Dockerfile not found.,NaN,NaN,125.570924,False,0.167677,456463,8785,lua + c,hard


## Overall Success Rate by Model

In [2]:
summary = (
    df.groupby("model")
    .agg(
        total=("success", "count"),
        successes=("success", "sum"),
        success_rate=("success", "mean"),
        mean_cost_usd=("cost_usd", "mean"),
        mean_agent_duration=("agent_duration_seconds", "mean"),
    )
    .sort_values("success_rate", ascending=False)
)

summary["success_rate"] = (summary["success_rate"] * 100).round(1)
summary["mean_cost_usd"] = summary["mean_cost_usd"].round(4)
summary["mean_agent_duration"] = summary["mean_agent_duration"].round(1)
summary

,total,successes,success_rate,mean_cost_usd,mean_agent_duration
model,,,,,
claude-opus,38,9,23.7,0.3408,876.9
claude-haiku,40,8,20.0,0.2531,671.2
codex-gpt-5.2,39,6,15.4,0.3361,790.9
codex-mini-gpt-5.1,41,0,0.0,0.0329,218.7


In [3]:
model_order = summary.index.tolist()  # already sorted best-to-worst

fig = px.bar(
    summary.reset_index(),
    x="model",
    y="success_rate",
    text="success_rate",
    color="model",
    title="Overall Success Rate by Model",
    labels={"success_rate": "Success Rate (%)", "model": "Model"},
    category_orders={"model": model_order},
)
fig.update_traces(texttemplate="%{text}%", textposition="outside")
fig.update_layout(yaxis_range=[0, 105], showlegend=False)
fig.show()

## Success Rate by Difficulty

In [4]:
diff_df = df.groupby(["difficulty", "model"])["success"].mean().reset_index()
diff_df["success"] = (diff_df["success"] * 100).round(1)

fig = px.bar(
    diff_df,
    x="difficulty",
    y="success",
    color="model",
    barmode="group",
    text="success",
    title="Success Rate by Difficulty & Model",
    labels={"success": "Success Rate (%)", "difficulty": "Difficulty"},
    category_orders={"model": model_order},
)
fig.update_traces(texttemplate="%{text}%", textposition="outside")
fig.update_layout(yaxis_range=[0, 105])
fig.show()

## Success Rate by Language

In [5]:
lang_df = df.groupby(["language", "model"])["success"].mean().reset_index()
lang_df["success"] = (lang_df["success"] * 100).round(1)

fig = px.bar(
    lang_df,
    x="language",
    y="success",
    color="model",
    barmode="group",
    text="success",
    title="Success Rate by Language & Model",
    labels={"success": "Success Rate (%)", "language": "Language"},
    category_orders={"model": model_order},
)
fig.update_traces(texttemplate="%{text}%", textposition="outside")
fig.update_layout(yaxis_range=[0, 105])
fig.show()

## Per-Repo Comparison (Heatmap)

In [6]:
pivot = df.pivot_table(index="repo", columns="model", values="success", aggfunc="mean")
pivot = pivot.sort_index()

fig = px.imshow(
    pivot,
    color_continuous_scale="RdYlGn",
    zmin=0,
    zmax=1,
    title="Success by Repo x Model",
    labels={"color": "Success Rate"},
    aspect="auto",
    height=max(400, len(pivot) * 22),
)
fig.show()

## Cost & Duration

In [7]:
fig = make_subplots(rows=1, cols=2, subplot_titles=("Cost vs Success", "Duration vs Success"))

for model in df["model"].unique():
    grp = df[df["model"] == model]
    jitter = np.random.uniform(-0.05, 0.05, len(grp))
    fig.add_trace(
        go.Scatter(
            x=grp["cost_usd"],
            y=grp["success"].astype(int) + jitter,
            mode="markers",
            name=model,
            opacity=0.6,
            marker={"size": 7},
            text=grp["repo"],
            hovertemplate="%{text}<br>Cost: $%{x:.4f}<br>Success: %{y:.0f}",
        ),
        row=1,
        col=1,
    )
    fig.add_trace(
        go.Scatter(
            x=grp["agent_duration_seconds"] / 60,
            y=grp["success"].astype(int) + jitter,
            mode="markers",
            name=model,
            opacity=0.6,
            showlegend=False,
            marker={"size": 7},
            text=grp["repo"],
            hovertemplate="%{text}<br>Duration: %{x:.1f} min<br>Success: %{y:.0f}",
        ),
        row=1,
        col=2,
    )

fig.update_xaxes(title_text="Cost (USD)", row=1, col=1)
fig.update_xaxes(title_text="Agent Duration (min)", row=1, col=2)
fig.update_yaxes(title_text="Success (jittered)", row=1, col=1)
fig.update_layout(height=400, width=900)
fig.show()

In [8]:
cost_summary = (
    df.groupby("model")
    .agg(
        mean_cost=("cost_usd", "mean"),
        median_cost=("cost_usd", "median"),
        mean_duration_min=("agent_duration_seconds", lambda x: (x / 60).mean()),
        median_duration_min=("agent_duration_seconds", lambda x: (x / 60).median()),
        timeouts=("agent_timed_out", "sum"),
    )
    .round(3)
)
cost_summary

,mean_cost,median_cost,mean_duration_min,median_duration_min,timeouts
model,,,,,
claude-haiku,0.253,0.301,11.187,10.922,0
claude-opus,0.341,0.000,14.615,15.716,0
codex-gpt-5.2,0.336,0.000,13.182,15.194,0
codex-mini-gpt-5.1,0.033,0.017,3.646,1.553,0


## Common Failure Reasons

In [9]:
failures = df[~df["success"]].copy()


def categorize_error(msg):
    if not msg:
        return "Unknown"
    m = msg.lower()
    # Agent never created required files
    if "dockerfile not found" in m:
        return "No files created"
    # Agent timeout (budget/time exhausted)
    if "timeout" in m or "timed out" in m or "status timeout" in m:
        return "Agent timeout"
    # Modal sandbox crashed / exited with non-zero status
    if "container id" in m and ("finished" in m or "status=" in m):
        return "Modal container crash"
    # Modal container disappeared before we could attach
    if "no container with id" in m:
        return "Modal container not found"
    # Docker image build failed (files exist but build broke)
    if "build failed" in m:
        return "Docker build failed"
    # Tests ran but returned non-zero
    if "test run failed" in m or ("test" in m and "return code" in m):
        return "Tests failed"
    # Low-level infra / network errors
    if any(x in m for x in ["nodename nor servname", "file descriptor not found", "errno", "eof"]):
        return "Infrastructure error"
    return "Other"


failures["error_category"] = failures["error_message"].apply(categorize_error)
err_df = failures.groupby(["model", "error_category"]).size().reset_index(name="count")

# Consistent color per category
category_colors = {
    "No files created": "#d62728",
    "Docker build failed": "#ff7f0e",
    "Tests failed": "#9467bd",
    "Agent timeout": "#e377c2",
    "Modal container crash": "#8c564b",
    "Modal container not found": "#bcbd22",
    "Infrastructure error": "#7f7f7f",
    "Unknown": "#c7c7c7",
    "Other": "#17becf",
}

fig = px.bar(
    err_df,
    x="model",
    y="count",
    color="error_category",
    title="Failure Reasons by Model",
    labels={"count": "Count", "error_category": "Error Category"},
    category_orders={"model": model_order},
    color_discrete_map=category_colors,
)
fig.show()

## Head-to-Head: Which repos does each model uniquely solve?

In [10]:
repo_success = df.groupby(["repo", "model"])["success"].any().unstack(fill_value=False)

for model in repo_success.columns:
    others = [c for c in repo_success.columns if c != model]
    unique = repo_success[model] & ~repo_success[others].any(axis=1)
    unique_repos = unique[unique].index.tolist()
    if unique_repos:
        print(f"\n{model} uniquely solves ({len(unique_repos)}): {', '.join(unique_repos)}")
    else:
        print(f"\n{model}: no uniquely solved repos")


claude-haiku uniquely solves (2): golangci-lint, starlette

claude-opus uniquely solves (1): fastapi

codex-gpt-5.2 uniquely solves (1): cupy

codex-mini-gpt-5.1: no uniquely solved repos
